In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess

def is_correct_repo() -> bool:
    try:
        result = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
        remote_url = result.stdout.strip()
        return remote_url in [
            "https://github.com/mariogemoll/reinforcement-learning.git",
            "git@github.com:mariogemoll/reinforcement-learning.git",
        ]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False


if not is_correct_repo():
    !git clone https://github.com/mariogemoll/reinforcement-learning.git

if not os.getcwd().endswith("reinforcement-learning/py"):
    %cd reinforcement-learning/py


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))


In [ ]:
%pip install -q gymnax

In [ ]:
import base64
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
from IPython.display import display
from tqdm.auto import tqdm

from rl.core.dqn import build_init_carry, fresh_params, get_chunk_runner
from rl.minatar.visualizations import MinAtarBreakoutVisualization
from rl.core.util import as_f32, plot_dqn_metrics, write_safetensors

In [ ]:
env_name = "Breakout-MinAtar"

# Base (best-known) hyperparameters from search
cfg = {
    "env_name": env_name,
    "arch": "cnn",
    "filters": [16, 32],
    "fc_dim": 128,
    "hidden_dim": 256,
    "num_layers": 1,
    "lr": 2e-4,
    "decay_dur": 400_000,
    "batch_size": 128,
    "buf_cap": 50_000,
    "learn_start": 2_000,
    "upd_every": 1000,
}

# Modes:
# - smoke: fastest sanity check (reduced model + short run)
# - quick: faster iteration with best model shape
# - full: original long run
# run_mode = "quick"  # "smoke" | "quick" | "full"
run_mode = "quick"

mode_overrides = {
    "smoke": {
        "filters": [8, 16],
        "fc_dim": 64,
        "buf_cap": 20_000,
        "batch_size": 64,
        "learn_start": 1_000,
        "total_steps": 100_000,
        "chunk_steps": 50_000,
    },
    "quick": {
        "total_steps": 300_000,
        "chunk_steps": 100_000,
    },
    "full": {
        "total_steps": 3_000_000,
        "chunk_steps": 100_000,
    },
}

if run_mode not in mode_overrides:
    raise ValueError(
        f"Unknown run_mode={run_mode!r}; expected one of {list(mode_overrides)}"
    )

selected = mode_overrides[run_mode]
cfg.update({k: v for k, v in selected.items() if k in cfg})
total_steps = int(selected["total_steps"])
chunk_steps = int(selected["chunk_steps"])
n_chunks = max(1, total_steps // chunk_steps)

print(
    f"Run mode: {run_mode}"
    f" | total_steps={total_steps:,} | chunk_steps={chunk_steps:,}"
    f" | chunks={n_chunks}"
)
print(
    f"Model: filters={cfg['filters']} fc_dim={cfg['fc_dim']}"
    f" batch={cfg['batch_size']} buf={cfg['buf_cap']:,}"
)

In [ ]:
# Train CNN DQN and plot metrics
import time

init_params = fresh_params(
    env_name=cfg["env_name"],
    arch="cnn",
    filters=tuple(cfg["filters"]),
    fc_dim=cfg["fc_dim"],
)

init_carry = build_init_carry(cfg, total_steps, init_params, jax.random.key(0))
runner = get_chunk_runner(
    chunk_steps, cfg, cfg["env_name"], cfg["hidden_dim"], cfg["num_layers"]
)

carry = init_carry
print(
    "Starting training. First chunk may take"
    " several minutes due to JAX/XLA compilation..."
)
train_start = time.perf_counter()

for chunk_idx in tqdm(range(n_chunks), desc="Training CNN DQN"):
    chunk_start = time.perf_counter()
    carry = runner(carry)
    jax.block_until_ready(carry["ep_count"])
    chunk_sec = time.perf_counter() - chunk_start
    if chunk_idx == 0:
        print(f"First chunk (compile + train) took {chunk_sec:.1f}s")
    carry = {
        **carry,
        "step_offset": carry["step_offset"] + jnp.int32(chunk_steps),
    }

total_sec = time.perf_counter() - train_start
print(f"Training finished in {total_sec/60:.1f} min")

ep_count = int(carry["ep_count"])
loss_count = int(carry["loss_idx"])
ep_rets = np.asarray(carry["ep_rets"])[:ep_count].tolist()
losses = np.asarray(carry["loss_arr"])[:loss_count].tolist()
best_params = carry["best_params"]

print(f"Episodes completed: {ep_count}")
print(f"Gradient updates: {loss_count}")
if ep_rets:
    print(f"Final episode return: {ep_rets[-1]:.2f}")
plot_dqn_metrics(losses, ep_rets)

In [ ]:
# Export trained CNN weights directly to safetensors
tensors = {}

conv_idx = 0
while True:
    name = f"conv_{conv_idx}"
    if name not in best_params:
        break
    tensors[f"{name}.kernel"] = as_f32(best_params[name]["kernel"])
    tensors[f"{name}.bias"] = as_f32(best_params[name]["bias"])
    conv_idx += 1

if conv_idx == 0:
    raise ValueError("No conv_* layers found in best_params")

for dense_name in ["fc", "out_layer"]:
    if dense_name not in best_params:
        raise ValueError(f"Missing layer in best_params: {dense_name}")
    tensors[f"{dense_name}.kernel"] = as_f32(best_params[dense_name]["kernel"])
    tensors[f"{dense_name}.bias"] = as_f32(best_params[dense_name]["bias"])

out_path = Path("dqn-minatar-breakout-cnn-weights.safetensors")
write_safetensors(out_path, tensors)
print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes)")
print("Exported keys:")
for key in sorted(tensors):
    print(" -", key, tensors[key].shape)

In [ ]:
%%bash
# Build MinAtar Breakout visualization
set -euo pipefail

cd ../ts

if command -v pnpm >/dev/null 2>&1; then
  echo "Using package manager: pnpm"
  pnpm i
  pnpm run build:anywidget-minatar-breakout
else
  echo "Using package manager: npm"
  npm install
  npm run build:anywidget-minatar-breakout
fi

In [ ]:
# Embed safetensors in anywidget and display
weights_bytes = Path("dqn-minatar-breakout-cnn-weights.safetensors").read_bytes()
weights_b64 = base64.b64encode(weights_bytes).decode("ascii")

display(MinAtarBreakoutVisualization(weights_base64=weights_b64))